# Lab 05-01 · Python Fundamentals for Engineers

Companion to chapter [05-01](../../chapters/05-01-python-fundamentals.md). Handout: [README.md](README.md).

Cells marked **🔮 Predict** ask for a prediction *before* running. Write it down (in a comment or on paper), then run the cell and explain any surprise. Part 5's timings depend on your computer; everything else is deterministic.

In [1]:
import math
import sys
import time
from fractions import Fraction
print(sys.version.split()[0])

3.13.18


## Part 1 — Predict the output

**🔮 Predict:** for each expression below, write the value Python will produce. Then run the cell.

In [2]:
expressions = [
    "7 / 2", "7 // 2", "-7 // 2", "7 % -2", "2 ** 0.5", "10 ** 20 + 1",
    "0.1 + 0.2", "round(0.5)", "round(1.5)", "int(3.99)",
    "'VDD' * 2", "'4.7k'[:-1]", "len('μA')", "[1, 2] * 2",
    "3 == 3.0", "0.1 * 3 == 0.3", "[1, 2] == [1, 2]", "[1, 2] is [1, 2]",
    "bool('')", "sorted(['10', '9', '100'])",
]
for e in expressions:
    print(f"{e:28} -> {eval(e)!r}")

7 / 2                        -> 3.5
7 // 2                       -> 3
-7 // 2                      -> -4
7 % -2                       -> -1
2 ** 0.5                     -> 1.4142135623730951
10 ** 20 + 1                 -> 100000000000000000001
0.1 + 0.2                    -> 0.30000000000000004
round(0.5)                   -> 0
round(1.5)                   -> 2
int(3.99)                    -> 3
'VDD' * 2                    -> 'VDDVDD'
'4.7k'[:-1]                  -> '4.7'
len('μA')                    -> 2
[1, 2] * 2                   -> [1, 2, 1, 2]
3 == 3.0                     -> True
0.1 * 3 == 0.3               -> False
[1, 2] == [1, 2]             -> True
[1, 2] is [1, 2]             -> False
bool('')                     -> False
sorted(['10', '9', '100'])   -> ['10', '100', '9']


Surprises worth understanding: `round(0.5)` and `round(1.5)` both round to the *even* neighbour (0 and 2; "banker's rounding", which avoids a systematic upward bias); `int(3.99)` truncates; `[1, 2] is [1, 2]` is `False` because two separate lists are built; and text sorts character by character, so `'100'` comes before `'9'`.

## Part 2 — Floats under the microscope

**🔮 Predict:** if you add 0.1 to a running total one million times, how far from 100000 will the result be? And what is the smallest positive `x` for which `1.0 + x != 1.0`?

In [3]:
total = 0.0
for _ in range(1_000_000):
    total += 0.1
print(f"loop: {total!r}, error {total - 100000:.4e}")
print(f"math.fsum: {math.fsum([0.1] * 1_000_000)!r}")
x = 1.0
while 1.0 + x / 2 != 1.0:
    x /= 2
print(f"smallest power of two that changes 1.0: {x!r} = 2**{round(math.log2(x))}; sys.float_info.epsilon = {sys.float_info.epsilon!r}")
print(Fraction(0.1), float(Fraction(0.1) - Fraction(1, 10)))

loop: 100000.00000133288, error 1.3329e-06
math.fsum: 100000.0
smallest power of two that changes 1.0: 2.220446049250313e-16 = 2**-52; sys.float_info.epsilon = 2.220446049250313e-16
3602879701896397/36028797018963968 5.551115123125783e-18


The loop is off by about $1.3\times10^{-6}$, far more than the representation error of 0.1 itself (one million times $5.6\times10^{-18}$): each addition rounds the growing total, and those roundings accumulate. `math.fsum` tracks the lost low-order bits and returns the correctly rounded sum, exactly 100000.0 here. The halving loop finds machine epsilon, $2^{-52}$: adding anything smaller (half of it rounds to the even neighbour) leaves 1.0 unchanged.

## Part 3 — Aliasing

**🔮 Predict:** after `grid[0][0] = 1`, how many ones does each grid contain?

In [4]:
shared = [[0] * 3] * 3
independent = [[0] * 3 for _ in range(3)]
shared[0][0] = 1
independent[0][0] = 1
print("shared:", shared, [id(row) for row in shared].count(id(shared[0])), "rows are the same object")
print("independent:", independent, len({id(row) for row in independent}), "distinct row objects")

shared: [[1, 0, 0], [1, 0, 0], [1, 0, 0]] 3 rows are the same object
independent: [[1, 0, 0], [0, 0, 0], [0, 0, 0]] 3 distinct row objects


`[[0] * 3] * 3` repeats a *reference* to one inner list three times. The comprehension evaluates `[0] * 3` once per row. `id()` returns an object's identity, so equal ids mean the same object.

## Part 4 — A resistor color-code decoder

The first two bands are digits and the third is a power-of-ten multiplier: yellow–violet–red means 47 × 10² = 4700 Ω.

**🔮 Predict:** what should the decoder do with `("blue", "grey", "purple")` (purple is not a color in the code)?

In [5]:
DIGITS = {"black": 0, "brown": 1, "red": 2, "orange": 3, "yellow": 4,
          "green": 5, "blue": 6, "violet": 7, "grey": 8, "white": 9}

def decode(bands):
    '''Resistance in ohms of a three-band resistor color code.'''
    if len(bands) != 3:
        raise ValueError("expected three bands")
    try:
        d1, d2, mult = (DIGITS[b.lower()] for b in bands)
    except KeyError as missing:
        raise ValueError(f"unknown color {missing}") from None
    return (10 * d1 + d2) * 10 ** mult

assert decode(("yellow", "violet", "red")) == 4700
assert decode(("brown", "black", "orange")) == 10_000
assert decode(("Brown", "Black", "Black")) == 10
try:
    decode(("blue", "grey", "purple"))
except ValueError as err:
    print("rejected:", err)
print("all tests passed;", decode(("yellow", "violet", "red")), decode(("brown", "black", "orange")))

rejected: unknown color 'purple'
all tests passed; 4700 10000


The `assert` statements are tests: each states a known answer, and the cell stops with an error if any fails (05-04 turns this habit into a test suite).

## Part 5 — Timing

**🔮 Predict:** rank three ways of summing the squares of the first million integers: an explicit loop, `sum` over a generator, and `sum` over a list comprehension.

In [6]:
N = 1_000_000

def loop_sum():
    total = 0
    for i in range(N):
        total += i * i
    return total

def gen_sum():
    return sum(i * i for i in range(N))

def list_sum():
    return sum([i * i for i in range(N)])

results = {}
for f in (loop_sum, gen_sum, list_sum):
    t0 = time.perf_counter()
    value = f()
    results[f.__name__] = time.perf_counter() - t0
    assert value == (N - 1) * N * (2 * N - 1) // 6     # all three must agree with the closed form
for name, seconds in results.items():
    print(f"{name:9} {seconds * 1e3:7.1f} ms")

loop_sum     71.7 ms
gen_sum      73.1 ms
list_sum     94.7 ms


On most machines the three are within a factor of about two of each other, and which one wins depends on the Python version and the machine (recent interpreters have made plain loops much faster). The lesson is that in pure Python the per-item interpreter overhead dominates however you write it. The real speedup, a factor of tens to hundreds, comes from moving the loop into compiled code with NumPy (05-02). The `assert` checks every method against the closed form $\sum i^2 = (N-1)N(2N-1)/6$ for $i < N$.

## Your turn

1. In Part 2, replace 0.1 with 0.5 (exactly representable in binary). Is the loop sum now exact? Why?
2. In Part 4, extend the decoder to four-band resistors whose fourth band gives the tolerance (gold ±5%, silver ±10%), returning a tuple `(ohms, tolerance)`.